# Entrenamiento de Modelos — Forest Cover Type

In [ ]:
import os
import pandas as pd
import numpy as np
import mysql.connector
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from utils.model_trainer import ModelTrainer

## 1. Conexión a MySQL y carga de datos procesados

In [ ]:
conn = mysql.connector.connect(
    host=os.getenv('MYSQL_HOST', 'mysql'),
    port=int(os.getenv('MYSQL_PORT', 3306)),
    user=os.getenv('MYSQL_USER', 'user'),
    password=os.getenv('MYSQL_PASSWORD', 'user1234'),
    database=os.getenv('MYSQL_DATABASE', 'mydatabase'),
)

df = pd.read_sql('SELECT * FROM processed_forest_cover', conn)
conn.close()

print(f'Registros cargados: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head()

## 2. Exploración rápida

In [ ]:
print('Valores nulos por columna:')
print(df.isnull().sum())
print(f'\nFilas duplicadas: {df.duplicated().sum()}')
print(f'\nDistribución de cover_type:\n{df["cover_type"].value_counts().sort_index()}')
print(f'\nWilderness areas: {df["wilderness_area"].unique()}')
print(f'Soil types: {sorted(df["soil_type"].unique())}')

## 3. Preparación de datos

In [ ]:
# Eliminar columnas de metadatos
drop_cols = ['id', 'group_id', 'ingestion_ts']
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Codificar columnas categóricas
le_wilderness = LabelEncoder()
le_soil = LabelEncoder()
df_clean['wilderness_area'] = le_wilderness.fit_transform(df_clean['wilderness_area'])
df_clean['soil_type'] = le_soil.fit_transform(df_clean['soil_type'])

X = df_clean.drop('cover_type', axis=1)
y = df_clean['cover_type']

print(f'Features: {list(X.columns)}')
print(f'Clases: {sorted(y.unique())}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Inicializar ModelTrainer (con MinIO)

In [ ]:
trainer = ModelTrainer(
    models_dir='/app/models',
    report_path='/app/report/model_metrics.csv',
    minio_endpoint=os.getenv('MINIO_ENDPOINT', 'minio:9000'),
    minio_access_key=os.getenv('MINIO_ACCESS_KEY', 'minio_user'),
    minio_secret_key=os.getenv('MINIO_SECRET_KEY', 'minio1234'),
    minio_bucket='mlmodels',
)

## 5. Entrenar Random Forest

In [ ]:
rf_metrics = trainer.train_and_save(
    name='randomforest',
    estimator=RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
    scaler=StandardScaler(),
)

## 6. Entrenar Gradient Boosting

In [ ]:
gb_metrics = trainer.train_and_save(
    name='gradientboosting',
    estimator=GradientBoostingClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42),
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
    scaler=StandardScaler(),
)

## 7. Entrenar SVM

In [ ]:
svm_metrics = trainer.train_and_save(
    name='svm',
    estimator=SVC(kernel='rbf', C=1.0, random_state=42),
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
    scaler=StandardScaler(),
)

## 8. Resumen de métricas

In [ ]:
report = pd.DataFrame([rf_metrics, gb_metrics, svm_metrics]).set_index('model')
print('Reporte guardado en /app/report/model_metrics.csv')
report